# **Import and Load Events**

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS

In [0]:
events = spark.table("events_delta")

# **Create Rating Mapping**
We convert Implicit behaviour into Explicit Rating 


In [0]:
interaction_df = events.withColumn(
    "rating",
    F.when(F.col("event_type") == "purchase", 3)
     .when(F.col("event_type") == "cart", 2)
     .otherwise(1)
).select("user_id", "product_id", "rating")

In [0]:
interaction_df.display()

**Ensure Integer IDs (Required)**

In [0]:
interaction_df = interaction_df \
    .withColumn("user_id", F.col("user_id").cast("int")) \
    .withColumn("product_id", F.col("product_id").cast("int")) \
    .withColumn("rating", F.col("rating").cast("float"))

In [0]:
import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlflow_tmp"

**Train ALS Model**

In [0]:
als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

als_model = als.fit(interaction_df)

# **Generate Top-5 Recommendations**

In [0]:
predictions = als_model.transform(interaction_df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("user_id").orderBy(desc("prediction"))

ranked_df = predictions.withColumn(
    "rank",
    row_number().over(window_spec)
)

In [0]:
top5_df = ranked_df.filter("rank <= 5") \
    .select(
        "user_id",
        "product_id",
        "prediction"
    )

In [0]:
final_recommendations = exploded_df.selectExpr(
    "user_id",
    "rec.product_id as recommended_product",
    "rec.rating as predicted_rating"
)

# **Flatten Recommendations**

In [0]:
top5_df = ranked_df.filter("rank <= 5") \
    .select(
        "user_id",
        "product_id",
        "prediction"
    )

In [0]:
top5_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_user_recommendations")

In [0]:
spark.sql("""
SELECT *
FROM gold_user_recommendations
LIMIT 20
""").display()